# LeetCode 1590 — Make Sum Divisible by P bằng Python

Link đề: https://leetcode.com/problems/make-sum-divisible-by-p/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minSubarray(self, nums: List[int], p: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến bài "xóa một đoạn để phần còn lại chia hết cho `p`" thành bài **"tìm đoạn ngắn nhất có tổng ≡ `k` (mod `p`)"**, với `k = sum(nums) % p`.
2. Viết baseline **thử mọi đoạn con** (O(n²) dùng tổng chạy) để làm chuẩn đối chiếu.
3. Nắm kỹ thuật **tổng tiền tố theo modulo** + **từ điển lưu vị trí xuất hiện GẦN NHẤT** của mỗi số dư — khuôn mẫu chung của rất nhiều bài "subarray sum".
4. Xử lý chính xác các chi tiết dễ sai: khởi tạo `{0: -1}`, lấy `% p` cho khóa cần tìm, **không được xóa cả mảng** (trả `-1`).
5. Biết tự kiểm thử: so sánh lời giải tối ưu với brute force trên hàng nghìn input ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho mảng số nguyên **dương** `nums` và số nguyên `p`. Hãy xóa đi **một đoạn con liên tiếp ngắn nhất** (có thể là đoạn rỗng — tức không xóa gì) sao cho **tổng các phần tử còn lại chia hết cho `p`**. **Không được phép xóa toàn bộ mảng.**

Trả về **độ dài** của đoạn ngắn nhất cần xóa, hoặc `-1` nếu không thể.

*Đoạn con (subarray)* là một khối các phần tử **liên tiếp** trong mảng.

**Ví dụ 1:** `nums = [3,1,4,2]`, `p = 6` → `1`. Tổng là 10, không chia hết cho 6. Xóa đoạn `[4]` thì còn `3+1+2 = 6` chia hết cho 6.

**Ví dụ 2:** `nums = [6,3,5,2]`, `p = 9` → `2`. Xóa 1 phần tử không đủ; xóa `[5,2]` còn `[6,3]` tổng 9.

**Ví dụ 3:** `nums = [1,2,3]`, `p = 3` → `0`. Tổng 6 đã chia hết cho 3, không cần xóa gì.

**Ràng buộc:**
- `1 ≤ nums.length ≤ 10⁵`
- `1 ≤ nums[i] ≤ 10⁹`
- `1 ≤ p ≤ 10⁹`

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [
    ([3, 1, 4, 2], 6, 1),
    ([6, 3, 5, 2], 9, 2),
    ([1, 2, 3], 3, 0),
]
for nums, p, want in examples:
    print(f"nums={nums}, p={p}, tổng={sum(nums)}, tổng % p = {sum(nums) % p} -> mong đợi {want}")

nums=[3, 1, 4, 2], p=6, tổng=10, tổng % p = 4 -> mong đợi 1
nums=[6, 3, 5, 2], p=9, tổng=16, tổng % p = 7 -> mong đợi 2
nums=[1, 2, 3], p=3, tổng=6, tổng % p = 0 -> mong đợi 0


Nhận xét nhanh từ dòng in trên: khi `tổng % p == 0` (ví dụ 3) thì đáp án là `0` ngay. Khi `tổng % p = k ≠ 0`, ta phải "lấy bớt đi" một phần có số dư đúng bằng `k`. Đó là toàn bộ chìa khóa — phần còn lại của notebook chỉ là làm việc đó **nhanh** và **không sai chi tiết**.

## 3. Kiến thức nền

### 3.1 Phép chia lấy dư `%` trong Python

- `a % p` luôn trả về số trong `[0, p-1]` khi `p > 0`, **kể cả khi `a` âm** (khác C++/Java, nơi `-1 % 6 == -1`).
- Tính chất quan trọng: `(a + b) % p == ((a % p) + (b % p)) % p` và `(a - b) % p == ((a % p) - (b % p)) % p`. Nhờ vậy ta có thể giữ tổng tiền tố ở dạng đã lấy dư, số luôn nhỏ.

In [2]:
print(10 % 6, (-1) % 6, (-7) % 6)          # Python: luôn không âm
a, b, p = 123456789, 987654321, 1000
print((a + b) % p, ((a % p) + (b % p)) % p)  # bằng nhau
print((a - b) % p, ((a % p) - (b % p)) % p)  # bằng nhau, kể cả khi hiệu âm

4 5 5
110 110
468 468


### 3.2 Tổng tiền tố (prefix sum)

Đặt `pre[0] = 0`, `pre[i] = nums[0] + … + nums[i-1]`. Khi đó tổng đoạn `nums[l..r]` (hai đầu bao gồm) là `pre[r+1] − pre[l]`. Một đoạn con tương ứng đúng **một cặp chỉ số** `(l, r+1)` trên mảng `pre`.

In [3]:
nums = [3, 1, 4, 2]
pre = [0] + list(accumulate(nums))
print("pre =", pre)
print("tổng nums[2..2] =", pre[3] - pre[2])
print("tổng nums[1..3] =", pre[4] - pre[1])

pre = [0, 3, 4, 8, 10]
tổng nums[2..2] = 4
tổng nums[1..3] = 7


### 3.3 Từ điển (`dict`) lưu vị trí

`dict` cho phép tra cứu/ghi trung bình O(1). Hai cách ghi có ý nghĩa khác nhau:
- `d[key] = i` — **ghi đè**, luôn giữ vị trí **gần nhất** (mới nhất).
- `d.setdefault(key, i)` — chỉ ghi lần đầu, giữ vị trí **xa nhất** (cũ nhất).

Bài này cần đoạn **ngắn nhất** nên muốn điểm bắt đầu càng gần càng tốt ⇒ dùng **ghi đè**.

In [4]:
d_new, d_old = {}, {}
for i, key in enumerate([2, 5, 2, 2, 5]):
    d_new[key] = i
    d_old.setdefault(key, i)
print("ghi đè (gần nhất):", d_new)
print("setdefault (xa nhất):", d_old)

ghi đè (gần nhất): {2: 3, 5: 4}
setdefault (xa nhất): {2: 0, 5: 1}


### 3.4 `float('inf')` / dùng `n` làm "vô cực"

Khi tìm min, ta thường khởi tạo `ans = n` (độ dài cả mảng). Vì không được xóa cả mảng, nếu cuối cùng `ans` vẫn là `n` thì nghĩa là **không có đoạn hợp lệ nào ngắn hơn** ⇒ trả `-1`. Mẹo này gộp hai việc (giá trị khởi tạo và điều kiện không hợp lệ) làm một.

In [5]:
n = 4
ans = n
for L in [3, 1, 2]:
    ans = min(ans, L)
print("ans =", ans, "->", -1 if ans == n else ans)

ans = 1 -> 1


## 4. Ý tưởng nền tảng / trực giác

Gọi `S = sum(nums)`, `k = S % p`. Xóa đoạn có tổng `X` thì phần còn lại là `S − X`. Ta cần:

```
(S − X) % p == 0   ⟺   X % p == S % p == k
```

Vậy bài toán trở thành: **tìm đoạn con ngắn nhất có tổng chia `p` dư `k`** (và độ dài `< n`).

**Tính tay ví dụ 2:** `nums = [6,3,5,2]`, `p = 9`, `S = 16`, `k = 7`.

| đoạn | tổng | tổng % 9 | = 7? |
|---|---|---|---|
| [6] | 6 | 6 | không |
| [3] | 3 | 3 | không |
| [5] | 5 | 5 | không |
| [2] | 2 | 2 | không |
| [6,3] | 9 | 0 | không |
| [3,5] | 8 | 8 | không |
| [5,2] | 7 | 7 | **có**, độ dài 2 |

Không có đoạn độ dài 1 nào dư 7, đoạn `[5,2]` dư 7 ⇒ đáp án 2. Khớp đề.

In [6]:
nums, p = [6, 3, 5, 2], 9
S = sum(nums); k = S % p
print("S =", S, " k =", k)
n = len(nums)
for L in range(1, n):
    for l in range(n - L + 1):
        seg = nums[l:l + L]
        if sum(seg) % p == k:
            print(f"độ dài {L}: đoạn {seg} có tổng {sum(seg)} ≡ {k} (mod {p})")

S = 16  k = 7
độ dài 2: đoạn [5, 2] có tổng 7 ≡ 7 (mod 9)


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** cố định điểm đầu `i`, mở rộng điểm cuối `j` và cộng dồn tổng chạy `s` (không cần tính lại tổng từ đầu). Với mỗi đoạn `nums[i..j]` có độ dài `< n`, nếu `(S − s) % p == 0` thì cập nhật đáp án.

Đừng quên trường hợp **đoạn rỗng** (độ dài 0): nếu `S % p == 0` thì trả `0` ngay.

In [7]:
def min_subarray_brute(nums, p):
    n = len(nums)
    S = sum(nums)
    if S % p == 0:
        return 0                      # đoạn rỗng
    best = -1
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            L = j - i + 1
            if L == n:                # không được xóa cả mảng
                continue
            if (S - s) % p == 0 and (best == -1 or L < best):
                best = L
    return best

for nums, p, want in examples:
    print(nums, p, "->", min_subarray_brute(nums, p), "| mong đợi", want)

[3, 1, 4, 2] 6 -> 1 | mong đợi 1
[6, 3, 5, 2] 9 -> 2 | mong đợi 2
[1, 2, 3] 3 -> 0 | mong đợi 0


**Độ phức tạp:** hai vòng lặp lồng nhau ⇒ **O(n²)** thời gian, **O(1)** bộ nhớ.

**Nút thắt:** với `n = 10⁵`, `n² = 10¹⁰` phép tính — quá chậm. Đoạn chương trình dưới đây đo thời gian thực để thấy tốc độ tăng gần gấp 4 khi `n` gấp đôi.

In [8]:
random.seed(0)
for n in [500, 1000, 2000]:
    arr = [random.randint(1, 10**9) for _ in range(n)]
    t = time.perf_counter()
    min_subarray_brute(arr, 10**9 - 7)
    print(f"n={n:5d}: {time.perf_counter() - t:.3f}s")

n=  500: 0.022s
n= 1000: 0.073s


n= 2000: 0.246s


## 6. Ý tưởng thuật toán / bước đột phá

Dùng tổng tiền tố: đoạn `nums[i..j]` có tổng `pre[j+1] − pre[i]`. Điều kiện:

```
(pre[j+1] − pre[i]) % p == k   ⟺   pre[i] % p == (pre[j+1] − k) % p
```

**Insight:** khi quét từ trái sang phải và đang đứng ở vị trí `j` với `cur = pre[j+1] % p`, ta **biết chính xác** số dư mà điểm bắt đầu cần có: `need = (cur − k) % p`. Chỉ cần hỏi "lần gần nhất tiền tố có số dư `need` là ở đâu?" — một phép tra `dict`.

**Quy tắc "nếu… thì…":**
- Nếu `S % p == 0` thì trả `0`.
- Nếu `need` đã có trong `last` thì đoạn `(last[need], j]` hợp lệ, độ dài `j − last[need]`.
- Sau đó **luôn** cập nhật `last[cur] = j` (ghi đè: vị trí càng gần càng cho đoạn ngắn).
- Khởi tạo `last = {0: -1}`: tiền tố rỗng có tổng 0 ở "vị trí −1", để đoạn bắt đầu từ chỉ số 0 cũng được xét.
- Cuối cùng nếu `ans == n` (chỉ có cả mảng là hợp lệ, hoặc không có gì) thì trả `-1`.

**Invariant:** trước khi xử lý vị trí `j`, `last[r]` là chỉ số **lớn nhất** `t < j` (tính cả `t = −1`) sao cho `pre[t+1] % p == r`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử theo độ dài tăng dần (cửa sổ trượt cố định độ dài)

Một cải tiến "thực dụng" của brute force: thay vì xét mọi đoạn rồi lấy min, ta duyệt **độ dài `L = 1, 2, …, n−1`** và với mỗi `L` trượt một cửa sổ độ dài `L` (cập nhật tổng O(1) mỗi bước). **Gặp độ dài đầu tiên có đoạn hợp lệ là dừng** — đó chắc chắn là đáp án nhỏ nhất.

- Ưu: thường dừng sớm khi đáp án nhỏ.
- Nhược: trường hợp xấu (đáp án lớn hoặc `-1`) vẫn O(n²).

Từng đoạn: (1) tính `S`, `k`; (2) với mỗi `L`, khởi tạo tổng cửa sổ đầu tiên; (3) trượt: cộng phần tử mới vào, trừ phần tử rơi ra; (4) kiểm tra `win % p == k`.

In [9]:
def min_subarray_by_length(nums, p):
    n = len(nums)
    k = sum(nums) % p
    if k == 0:
        return 0
    for L in range(1, n):             # L = n bị cấm
        win = sum(nums[:L])
        if win % p == k:
            return L
        for r in range(L, n):         # cửa sổ mới [r-L+1 .. r]
            win += nums[r] - nums[r - L]
            if win % p == k:
                return L
    return -1

for nums, p, want in examples:
    print(nums, p, "->", min_subarray_by_length(nums, p), "| mong đợi", want)

[3, 1, 4, 2] 6 -> 1 | mong đợi 1
[6, 3, 5, 2] 9 -> 2 | mong đợi 2
[1, 2, 3] 3 -> 0 | mong đợi 0


Code nộp LeetCode của Cách 2 (đúng nhưng có thể TLE ở test xấu — chỉ để so sánh):

```python
class Solution:
    def minSubarray(self, nums: List[int], p: int) -> int:
        n = len(nums)
        k = sum(nums) % p
        if k == 0:
            return 0
        for L in range(1, n):
            win = sum(nums[:L])
            if win % p == k:
                return L
            for r in range(L, n):
                win += nums[r] - nums[r - L]
                if win % p == k:
                    return L
        return -1
```

**Độ phức tạp:** Time O(n · ans) nếu tìm thấy (O(n²) nếu `-1`), Space O(1).

### 7.2 Cách 3 — Tiền tố modulo + `dict` vị trí gần nhất (tối ưu)

Giải thích từng đoạn:

1. `k = sum(nums) % p`; nếu `k == 0` trả `0`.
2. `last = {0: -1}` — tiền tố rỗng.
3. Duyệt `j, x`: `cur = (cur + x) % p` — giữ tổng tiền tố ở dạng số dư (số nhỏ, trong ngôn ngữ khác tránh tràn số).
4. `need = (cur - k) % p` — **phải** lấy `% p`, vì `cur - k` có thể âm hoặc không nằm trong `[0, p)`.
5. Nếu `need in last`: `ans = min(ans, j - last[need])`.
6. `last[cur] = j` — ghi đè.
7. Trả `-1 if ans == n else ans`.

Lưu ý thứ tự bước 5 và 6: **tra trước rồi mới ghi**. Nếu ghi trước, ta có thể ghép `j` với chính nó, tạo đoạn rỗng độ dài 0. Ở bài này `k ≠ 0` nên `need ≠ cur` và thứ tự không gây sai, nhưng ở các biến thể (ví dụ `k = 0`, bài tập 5) thì có — hãy giữ thói quen "tra rồi ghi".

In [10]:
def min_subarray_hash(nums, p):
    n = len(nums)
    k = sum(nums) % p
    if k == 0:
        return 0
    last = {0: -1}
    cur = 0
    ans = n
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        need = (cur - k) % p
        if need in last:
            ans = min(ans, j - last[need])
        last[cur] = j
    return -1 if ans == n else ans

for nums, p, want in examples:
    print(nums, p, "->", min_subarray_hash(nums, p), "| mong đợi", want)

[3, 1, 4, 2] 6 -> 1 | mong đợi 1
[6, 3, 5, 2] 9 -> 2 | mong đợi 2
[1, 2, 3] 3 -> 0 | mong đợi 0


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def minSubarray(self, nums: List[int], p: int) -> int:
        n = len(nums)
        k = sum(nums) % p
        if k == 0:
            return 0
        last = {0: -1}
        cur, ans = 0, n
        for j, x in enumerate(nums):
            cur = (cur + x) % p
            need = (cur - k) % p
            if need in last:
                ans = min(ans, j - last[need])
            last[cur] = j
        return -1 if ans == n else ans
```

**Độ phức tạp:** Time **O(n)** trung bình (mỗi phần tử một lần tra + một lần ghi `dict`), Space **O(min(n, p))** cho `dict`.

### 7.3 Cách 4 — Biến thể không cần `% p` ở bước tra: tra theo "tiền tố cần tìm" bằng mảng `pre` đầy đủ

Đây là cùng ý tưởng Cách 3 nhưng viết theo kiểu "tính hết `pre` trước rồi mới quét". Nó dài hơn một chút và tốn O(n) bộ nhớ cho `pre`, nhưng giúp người mới **nhìn rõ** công thức `pre[i] ≡ pre[j+1] − k`. Ta cũng thấy rõ tại sao chỉ số trong `dict` là `i` (điểm bắt đầu) và độ dài là `(j+1) − i`.

In [11]:
def min_subarray_prefix_array(nums, p):
    n = len(nums)
    pre = [0] * (n + 1)
    for i, x in enumerate(nums):
        pre[i + 1] = (pre[i] + x) % p
    k = pre[n]
    if k == 0:
        return 0
    last = {}
    ans = n
    for e in range(n + 1):            # e = j+1, chỉ số kết thúc (không bao gồm)
        need = (pre[e] - k) % p
        if need in last:
            ans = min(ans, e - last[need])
        last[pre[e]] = e              # i = e có thể làm điểm bắt đầu về sau
    return -1 if ans == n else ans

for nums, p, want in examples:
    print(nums, p, "->", min_subarray_prefix_array(nums, p), "| mong đợi", want)

[3, 1, 4, 2] 6 -> 1 | mong đợi 1
[6, 3, 5, 2] 9 -> 2 | mong đợi 2
[1, 2, 3] 3 -> 0 | mong đợi 0


Ở Cách 4, ta ghi `last[pre[e]] = e` **sau khi** tra, nên với `e = 0` thì `dict` rỗng lúc tra (không tạo đoạn rỗng), rồi ghi `last[0] = 0`. Đây chính là `{0: -1}` của Cách 3 sau khi dịch chỉ số đi 1.

**Độ phức tạp:** Time O(n), Space O(n).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1: `nums = [3,1,4,2]`, `p = 6`, `S = 10`, `k = 4`.

| j | x | cur | need = (cur−4)%6 | need ∈ last? | ans | last sau khi ghi |
|---|---|---|---|---|---|---|
| — | — | 0 | — | — | 4 | {0:−1} |
| 0 | 3 | 3 | 5 | không | 4 | {0:−1, 3:0} |
| 1 | 1 | 4 | 0 | có, last[0]=−1 → độ dài 2 | 2 | {0:−1, 3:0, 4:1} |
| 2 | 4 | 2 | 4 | có, last[4]=1 → độ dài 1 | 1 | {…, 2:2} |
| 3 | 2 | 4 | 0 | có, last[0]=−1 → độ dài 4 | 1 | {…, 4:3} |

Đáp án 1 (đoạn `nums[2..2] = [4]`). Ở `j = 1`, đoạn `[3,1]` dài 2 cũng hợp lệ (tổng 4 ≡ 4), nhưng sau đó tìm được đoạn ngắn hơn.

In [12]:
def trace_hash(nums, p):
    n = len(nums)
    k = sum(nums) % p
    print(f"S={sum(nums)}, k={k}")
    if k == 0:
        print("k == 0 -> trả 0"); return 0
    last = {0: -1}; cur = 0; ans = n
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        need = (cur - k) % p
        found = need in last
        if found:
            L = j - last[need]
            ans = min(ans, L)
        msg = f"đoạn ({last[need]}, {j}] dài {L}" if found else "không"
        last[cur] = j
        print(f"j={j} x={x} cur={cur} need={need} tìm thấy: {msg:22s} ans={ans} last={last}")
    res = -1 if ans == n else ans
    print("=> kết quả", res)
    return res

trace_hash([3, 1, 4, 2], 6)

S=10, k=4
j=0 x=3 cur=3 need=5 tìm thấy: không                  ans=4 last={0: -1, 3: 0}
j=1 x=1 cur=4 need=0 tìm thấy: đoạn (-1, 1] dài 2     ans=2 last={0: -1, 3: 0, 4: 1}
j=2 x=4 cur=2 need=4 tìm thấy: đoạn (1, 2] dài 1      ans=1 last={0: -1, 3: 0, 4: 1, 2: 2}
j=3 x=2 cur=4 need=0 tìm thấy: đoạn (-1, 3] dài 4     ans=1 last={0: -1, 3: 0, 4: 3, 2: 2}
=> kết quả 1


1

In [13]:
# Trace ví dụ 2 và một ca phải trả -1 (chỉ cả mảng mới có tổng dư k)
trace_hash([6, 3, 5, 2], 9)
print()
trace_hash([1, 2, 3], 7)

S=16, k=7
j=0 x=6 cur=6 need=8 tìm thấy: không                  ans=4 last={0: -1, 6: 0}
j=1 x=3 cur=0 need=2 tìm thấy: không                  ans=4 last={0: 1, 6: 0}
j=2 x=5 cur=5 need=7 tìm thấy: không                  ans=4 last={0: 1, 6: 0, 5: 2}
j=3 x=2 cur=7 need=0 tìm thấy: đoạn (1, 3] dài 2      ans=2 last={0: 1, 6: 0, 5: 2, 7: 3}
=> kết quả 2

S=6, k=6
j=0 x=1 cur=1 need=2 tìm thấy: không                  ans=3 last={0: -1, 1: 0}
j=1 x=2 cur=3 need=4 tìm thấy: không                  ans=3 last={0: -1, 1: 0, 3: 1}
j=2 x=3 cur=6 need=0 tìm thấy: đoạn (-1, 2] dài 3     ans=3 last={0: -1, 1: 0, 3: 1, 6: 2}
=> kết quả -1


-1

Ở ca `[1,2,3]`, `p = 7`: `k = 6`, chỉ khi `j = 2` mới tìm thấy `need = 0` ở vị trí `−1`, độ dài `3 == n` ⇒ đó là xóa cả mảng ⇒ không hợp lệ ⇒ `-1`.

## 9. Code nộp LeetCode

```python
from typing import List

class Solution:
    def minSubarray(self, nums: List[int], p: int) -> int:
        n = len(nums)
        k = sum(nums) % p
        if k == 0:
            return 0
        last = {0: -1}          # số dư tiền tố -> chỉ số gần nhất
        cur, ans = 0, n
        for j, x in enumerate(nums):
            cur = (cur + x) % p
            need = (cur - k) % p
            if need in last:
                ans = min(ans, j - last[need])
            last[cur] = j
        return -1 if ans == n else ans
```

In [14]:
class Solution:
    def minSubarray(self, nums: List[int], p: int) -> int:
        n = len(nums)
        k = sum(nums) % p
        if k == 0:
            return 0
        last = {0: -1}
        cur, ans = 0, n
        for j, x in enumerate(nums):
            cur = (cur + x) % p
            need = (cur - k) % p
            if need in last:
                ans = min(ans, j - last[need])
            last[cur] = j
        return -1 if ans == n else ans

sol = Solution()
for nums, p, want in examples:
    got = sol.minSubarray(nums, p)
    print(nums, p, "->", got, "OK" if got == want else "SAI")

[3, 1, 4, 2] 6 -> 1 OK
[6, 3, 5, 2] 9 -> 2 OK
[1, 2, 3] 3 -> 0 OK


## 10. Vì sao thuật toán đúng

1. **Quy đổi đúng:** xóa đoạn tổng `X` ⇒ còn `S − X`; `(S − X) ≡ 0 ⟺ X ≡ k (mod p)`. Mọi đoạn hợp lệ đều là một đoạn có tổng ≡ `k`, và ngược lại.
2. **Mọi đoạn đều được "nhìn thấy":** đoạn `nums[i..j]` tương ứng cặp tiền tố `(i−1, j)` theo chỉ số của `last` (với `−1` là tiền tố rỗng). Khi đứng ở `j`, ta xét điểm bắt đầu tốt nhất cho **mọi** đoạn kết thúc tại `j`.
3. **Chọn điểm gần nhất là tối ưu:** với `j` cố định, độ dài `j − t` nhỏ nhất khi `t` lớn nhất. Invariant đảm bảo `last[need]` là `t` lớn nhất `< j` có số dư `need`. Nên ở mỗi `j` ta có đoạn ngắn nhất kết thúc tại `j`; lấy min theo mọi `j` là đoạn ngắn nhất toàn cục.
4. **Điều kiện không xóa cả mảng:** đoạn dài `n` chỉ có thể là cả mảng. Vì `ans` khởi tạo bằng `n` và chỉ giảm, `ans == n` cuối cùng nghĩa là chỉ có cả mảng (hoặc không đoạn nào) thỏa ⇒ `-1`.
5. **Đoạn rỗng** chỉ hợp lệ khi `k == 0` — đã xử lý riêng, trả 0.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | O(n²) | O(1) | luôn duyệt hết |
| 2. Theo độ dài | O(n · ans), worst O(n²) | O(1) | dừng sớm nếu đáp án nhỏ |
| 3. Tiền tố mod + dict | O(n) trung bình | O(min(n, p)) | tối ưu |
| 4. Mảng pre + dict | O(n) | O(n) | dễ đọc công thức |

`dict` có độ phức tạp trung bình O(1); worst case lý thuyết O(n) mỗi thao tác nhưng với khóa là số nguyên nhỏ thì thực tế không xảy ra. Số khóa khác nhau ≤ `min(n + 1, p)`.

In [15]:
random.seed(1)
arr = [random.randint(1, 10**9) for _ in range(10**5)]
for f in [min_subarray_hash, min_subarray_prefix_array]:
    t = time.perf_counter()
    r = f(arr, 10**9 - 7)
    print(f"{f.__name__:28s} n=1e5 -> {r}, {time.perf_counter() - t:.3f}s")

min_subarray_hash            n=1e5 -> 1329, 0.031s
min_subarray_prefix_array    n=1e5 -> 1329, 0.034s


## 12. Bộ test

Gồm: ví dụ đề, các edge case (1 phần tử, `p = 1`, `p` lớn hơn tổng, chỉ cả mảng thỏa, đáp án là tiền tố/hậu tố), và đối chiếu ngẫu nhiên giữa 4 cách.

In [16]:
ALL = [min_subarray_brute, min_subarray_by_length, min_subarray_hash,
       min_subarray_prefix_array, lambda a, p: Solution().minSubarray(a, p)]

cases = examples + [
    ([5], 5, 0),            # 1 phần tử đã chia hết
    ([5], 3, -1),           # 1 phần tử, không thể (không được xóa cả mảng)
    ([7, 8, 9], 1, 0),      # p = 1: mọi số chia hết
    ([1, 2, 3], 7, -1),     # chỉ cả mảng có tổng ≡ k
    ([1, 2, 3], 100, -1),   # p lớn hơn tổng
    ([4, 1, 1], 5, 0),      # CỐ Ý ghi sai kỳ vọng (đúng là 1) để minh họa
    ([8, 5], 2, 1),         # xóa [5]
    ([1000000000, 1000000000, 1000000000], 3, 0),
    ([2, 3, 4, 5], 7, 0),   # tổng 14
    ([9, 1, 1, 1], 3, 2),   # CỐ Ý ghi sai kỳ vọng (đúng là 0) để minh họa
]
# Tính lại kỳ vọng bằng brute force — thói quen kiểm chứng ca tự nghĩ ra:
fixed = []
for a, p, w in cases:
    truth = min_subarray_brute(a, p)
    if truth != w:
        print(f"Sửa kỳ vọng {a}, p={p}: {w} -> {truth}")
    fixed.append((a, p, truth))
for a, p, w in fixed:
    for f in ALL:
        assert f(a, p) == w, (f, a, p)
print("Tất cả", len(fixed), "ca cố định đều khớp cho", len(ALL), "cách.")

Sửa kỳ vọng [4, 1, 1], p=5: 0 -> 1
Sửa kỳ vọng [9, 1, 1, 1], p=3: 2 -> 0
Tất cả 13 ca cố định đều khớp cho 5 cách.


Bài học nhỏ ở ô trên: **đừng tin kỳ vọng ghi tay** cho các ca tự nghĩ ra — luôn kiểm chứng bằng brute force. (Ô in ra hai dòng "Sửa kỳ vọng" vì ta cố ý ghi sai để minh họa.)

Tiếp theo: đối chiếu ngẫu nhiên. Ràng buộc đề cho phép `n` tới `10⁵` và giá trị tới `10⁹`; để brute force chạy nhanh ta thử `n ≤ 12`, giá trị nhỏ và `p` nhỏ (để có nhiều trường hợp chia hết thú vị), rồi thêm một đợt giá trị lớn.

In [17]:
random.seed(2024)
cnt = 0
for _ in range(3000):
    n = random.randint(1, 12)
    a = [random.randint(1, 20) for _ in range(n)]
    p = random.randint(1, 15)
    want = min_subarray_brute(a, p)
    for f in ALL:
        assert f(a, p) == want, (f, a, p)
    cnt += 1
for _ in range(500):
    n = random.randint(1, 30)
    a = [random.randint(1, 10**9) for _ in range(n)]
    p = random.randint(1, 10**9)
    want = min_subarray_brute(a, p)
    for f in ALL:
        assert f(a, p) == want
    cnt += 1
print("Đã đối chiếu", cnt, "bộ ngẫu nhiên: khớp hoàn toàn.")

Đã đối chiếu 3500 bộ ngẫu nhiên: khớp hoàn toàn.


In [18]:
# Thống kê phân bố đáp án để chắc bộ test phủ đủ các nhánh (0, -1, và các độ dài khác)
from collections import Counter
random.seed(7)
dist = Counter()
for _ in range(2000):
    n = random.randint(1, 8)
    a = [random.randint(1, 9) for _ in range(n)]
    p = random.randint(1, 12)
    dist[min_subarray_hash(a, p)] += 1
print(sorted(dist.items()))

[(-1, 520), (0, 501), (1, 711), (2, 183), (3, 57), (4, 18), (5, 5), (6, 2), (7, 3)]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên luật "không được xóa cả mảng"

Code sai trả về `ans` mà không kiểm tra `ans == n`, hoặc khởi tạo `ans = float('inf')` rồi để đoạn dài `n` lọt vào.

In [19]:
def wrong_allow_whole(nums, p):
    n = len(nums)
    k = sum(nums) % p
    if k == 0:
        return 0
    last = {0: -1}; cur = 0; ans = float('inf')
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        need = (cur - k) % p
        if need in last:
            ans = min(ans, j - last[need])
        last[cur] = j
    return -1 if ans == float('inf') else ans

a, p = [1, 2, 3], 7
print("sai :", wrong_allow_whole(a, p), "| đúng:", min_subarray_hash(a, p))

sai : 3 | đúng: -1


**Vì sao sai:** với `[1,2,3]`, `p = 7`, chỉ có cả mảng (tổng 6) là có số dư 6, code sai trả `3` — tức xóa hết, trái đề. **Sửa:** khởi tạo `ans = n` và trả `-1 if ans == n else ans` (như Cách 3).

### Lỗi 2 — Dùng vị trí xuất hiện ĐẦU TIÊN (`setdefault`) thay vì gần nhất

In [20]:
def wrong_first_index(nums, p):
    n = len(nums)
    k = sum(nums) % p
    if k == 0:
        return 0
    last = {0: -1}; cur = 0; ans = n
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        need = (cur - k) % p
        if need in last:
            ans = min(ans, j - last[need])
        last.setdefault(cur, j)        # SAI: giữ vị trí cũ nhất
    return -1 if ans == n else ans

a, p = [8, 5], 2
print("sai :", wrong_first_index(a, p), "| đúng:", min_subarray_hash(a, p))

sai : -1 | đúng: 1


**Vì sao sai:** sau phần tử `8`, tiền tố dư 0 xuất hiện lại ở vị trí 0, nhưng `setdefault` vẫn giữ `−1`. Ở `j = 1` cần `need = 0` → lấy `−1` → đoạn dài 2 = cả mảng → trả `-1`, trong khi xóa `[5]` (dài 1) là đủ. Muốn đoạn **ngắn nhất** thì điểm bắt đầu phải **gần nhất** ⇒ dùng `last[cur] = j`.

### Lỗi 3 — Quên khởi tạo `{0: -1}`

In [21]:
def wrong_no_init(nums, p):
    n = len(nums)
    k = sum(nums) % p
    if k == 0:
        return 0
    last = {}; cur = 0; ans = n         # SAI: thiếu tiền tố rỗng
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        need = (cur - k) % p
        if need in last:
            ans = min(ans, j - last[need])
        last[cur] = j
    return -1 if ans == n else ans

a, p = [9, 7], 7
print("sai :", wrong_no_init(a, p), "| đúng:", min_subarray_hash(a, p))

sai : -1 | đúng: 1


**Vì sao sai:** `S = 16`, `k = 2`. Đoạn cần xóa là `[9]` (9 ≡ 2 mod 7) — đoạn **bắt đầu từ chỉ số 0**, ứng với điểm bắt đầu "tiền tố rỗng" ở vị trí `−1`. Thiếu `{0: -1}` thì không bao giờ tìm được các đoạn là tiền tố của mảng.

### Lỗi 4 — Quên `% p` khi tính số dư cần tìm

In [22]:
def wrong_no_mod(nums, p):
    n = len(nums)
    k = sum(nums) % p
    if k == 0:
        return 0
    last = {0: -1}; cur = 0; ans = n
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        need = cur - k                 # SAI: có thể âm
        if need in last:
            ans = min(ans, j - last[need])
        last[cur] = j
    return -1 if ans == n else ans

a, p = [9, 4, 3], 6
print("sai :", wrong_no_mod(a, p), "| đúng:", min_subarray_hash(a, p))

sai : -1 | đúng: 1


**Vì sao sai:** `S = 16`, `k = 4`, các `cur` lần lượt là `3, 1, 4`. Ở `j = 1`, `cur = 1`, số dư cần tìm đúng phải là `(1 − 4) % 6 = 3` — có trong `last` (vị trí 0) ⇒ đoạn `[4]` dài 1. Code sai lại tìm khóa `1 − 4 = −3`, mà khóa trong `last` luôn thuộc `[0, p)`, nên trượt. Kết quả: bỏ sót **mọi** đoạn kết thúc tại vị trí có `cur < k`. Ở `j = 2` chỉ còn tìm thấy đoạn dài 3 = cả mảng ⇒ trả `-1`. **Sửa:** `need = (cur - k) % p`.

### Lỗi 5 — (ngôn ngữ khác) tràn số khi cộng dồn

Trong Python số nguyên không tràn, nhưng nếu dịch sang C++/Java mà cộng `sum` trực tiếp (tới `10⁵ · 10⁹ = 10¹⁴`) bằng `int` thì tràn. Giữ `cur = (cur + x) % p` ngay từ đầu là thói quen an toàn — và trong Python cũng giúp số nhỏ, tra `dict` nhanh hơn.

In [23]:
# Minh họa tràn số 32-bit bằng cách mô phỏng phép cộng int32
def add_int32(a, b):
    s = (a + b) & 0xFFFFFFFF
    return s - (1 << 32) if s >= (1 << 31) else s

tot = 0
for x in [10**9] * 3:
    tot = add_int32(tot, x)
print("tổng int32 (tràn):", tot, "| tổng thật:", 3 * 10**9)

tổng int32 (tràn): -1294967296 | tổng thật: 3000000000


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"subarray" + "sum" + "divisible / modulo k"** ⇒ nghĩ ngay **tiền tố modulo + hash map**.
- Hỏi **ngắn nhất** ⇒ lưu chỉ số **gần nhất** (ghi đè). Hỏi **dài nhất** ⇒ lưu chỉ số **đầu tiên** (`setdefault`). Hỏi **đếm số đoạn** ⇒ lưu **số lần xuất hiện**.
- Luôn tự hỏi: "tiền tố rỗng" đã có trong map chưa? (`{0: -1}` cho chỉ số, `{0: 1}` cho đếm).
- Bài "xóa đoạn để phần còn lại thỏa X" thường quy về "tìm đoạn có tính chất bù của X" (ở đây: tổng ≡ `k`).
- Đọc kỹ các ràng buộc ngầm: "possibly empty", "not allowed to remove the whole array".

## 15. Biến thể của bài

1. **LC 974 — Subarray Sums Divisible by K:** đếm số đoạn có tổng chia hết cho `k` ⇒ map đếm số lần xuất hiện của mỗi số dư.
2. **LC 523 — Continuous Subarray Sum:** có đoạn độ dài ≥ 2 tổng là bội của `k` không ⇒ map lưu chỉ số **đầu tiên**.
3. **LC 560 — Subarray Sum Equals K:** đếm đoạn có tổng **bằng** `k` (không modulo, có số âm).
4. **LC 325 — Maximum Size Subarray Sum Equals k:** dài nhất ⇒ chỉ số đầu tiên.
5. **Giữ lại thay vì xóa:** "tìm đoạn con DÀI NHẤT có tổng chia hết cho p" — đổi chiều: giữ đoạn liên tiếp dài nhất ⇔ (nếu đề yêu cầu giữ một đoạn) dùng chỉ số đầu tiên.
6. **Xóa đoạn để phần còn lại ≡ `t` (mod p)** với `t` bất kỳ: chỉ thay `k = (S − t) % p`.

In [24]:
# Minh họa biến thể 1 (LC 974): đếm số đoạn có tổng chia hết cho k
from collections import defaultdict
def subarrays_div_by_k(nums, k):
    cnt = defaultdict(int); cnt[0] = 1
    cur = res = 0
    for x in nums:
        cur = (cur + x) % k
        res += cnt[cur]
        cnt[cur] += 1
    return res
print(subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5))   # LC 974 ví dụ: 7

7


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi đoạn với tổng chạy | O(n²) | O(1) | làm chuẩn đối chiếu, `n` ≤ vài nghìn |
| 2. Theo độ dài | tăng `L`, cửa sổ trượt, gặp là dừng | O(n·ans) / O(n²) | O(1) | khi biết đáp án thường nhỏ |
| 3. Tiền tố mod + dict | tra số dư cần tìm, lưu chỉ số gần nhất | O(n) | O(min(n,p)) | **nộp bài** |
| 4. Mảng pre + dict | như 3 nhưng tính `pre` trước | O(n) | O(n) | học / giải thích công thức |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Không chạy code, đoán `minSubarray` cho: (a) `[1,1,1]`, `p=2`; (b) `[4,4,2]`, `p=3`; (c) `[2,2]`, `p=4`.

**Bài 2 (tính tay):** Với `nums = [6,3,5,2]`, `p = 9`, lập bảng `j, cur, need, last` như mục 8 và chỉ ra ở bước nào `ans` nhận giá trị 2.

**Bài 3 (điền TODO):** Hoàn thiện hàm `count_div_subarrays(nums, k)` đếm số đoạn con có tổng chia hết cho `k` (LC 974).

**Bài 4 (tự viết lại):** Viết `min_subarray_with_range(nums, p)` trả về bộ `(độ_dài, l, r)` — đoạn ngắn nhất cần xóa (chỉ số bao gồm), hoặc `(-1, -1, -1)`; nếu không cần xóa trả `(0, -1, -1)`. Khi có nhiều đoạn ngắn nhất, trả đoạn kết thúc sớm nhất.

**Bài 5 (mini challenge):** Viết `shortest_divisible_subarray(nums, p)` — độ dài đoạn con **không rỗng** ngắn nhất có tổng **chia hết** cho `p` (cho phép là cả mảng), hoặc `-1` nếu không có.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_div_subarrays(nums, k):
    cnt = {0: 1}     # tiền tố rỗng
    cur = 0
    res = 0
    for x in nums:
        # TODO: cập nhật cur theo modulo k
        # TODO: cộng vào res số lần số dư cur đã xuất hiện
        # TODO: tăng cnt[cur]
        pass
    return res

In [26]:
# Bài 4, 5 — khung để tự viết (chỉ định nghĩa, không gọi)
def min_subarray_with_range(nums, p):
    # TODO
    pass

def shortest_divisible_subarray(nums, p):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính `k = S % p` trước. Mức 2: tìm đoạn ngắn nhất tổng ≡ `k`. Mức 3: (a) `S=3`, `k=1` → xóa một số 1; (b) `S=10`, `k=1` → số nào ≡ 1 mod 3? (`4`); (c) `S=4`, `k=0`.

**Bài 2.** Mức 1: `k = 7`. Mức 2: `cur` lần lượt là `6, 0, 5, 7`. Mức 3: ở `j = 3`, `need = 0`, `last[0] = 1` ⇒ độ dài 2.

**Bài 3.** Mức 1: giống LC 974. Mức 2: `cur = (cur + x) % k`. Mức 3: `res += cnt.get(cur, 0)` rồi `cnt[cur] = cnt.get(cur, 0) + 1`.

**Bài 4.** Mức 1: dựa trên Cách 3. Mức 2: khi tìm được độ dài nhỏ hơn hẳn thì lưu `l = last[need] + 1`, `r = j`. Mức 3: dùng `<` (không phải `<=`) để giữ đoạn kết thúc sớm nhất.

**Bài 5.** Mức 1: tổng chia hết cho `p` ⇔ hai tiền tố cùng số dư. Mức 2: `need = cur` (vì `k = 0`), vẫn lưu chỉ số gần nhất. Mức 3: không có điều kiện "≠ cả mảng"; khởi tạo `ans = n + 1`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = [Solution().minSubarray(a, p) for a, p in [([1, 1, 1], 2), ([4, 4, 2], 3), ([2, 2], 4)]]
print("Bài 1:", ans1)
assert ans1 == [1, 1, 0]

# Bài 2: in bảng bằng trace
trace_hash([6, 3, 5, 2], 9)

Bài 1: [1, 1, 0]
S=16, k=7
j=0 x=6 cur=6 need=8 tìm thấy: không                  ans=4 last={0: -1, 6: 0}
j=1 x=3 cur=0 need=2 tìm thấy: không                  ans=4 last={0: 1, 6: 0}
j=2 x=5 cur=5 need=7 tìm thấy: không                  ans=4 last={0: 1, 6: 0, 5: 2}
j=3 x=2 cur=7 need=0 tìm thấy: đoạn (1, 3] dài 2      ans=2 last={0: 1, 6: 0, 5: 2, 7: 3}
=> kết quả 2


2

In [28]:
# Bài 3
def count_div_subarrays_sol(nums, k):
    cnt = {0: 1}
    cur = res = 0
    for x in nums:
        cur = (cur + x) % k
        res += cnt.get(cur, 0)
        cnt[cur] = cnt.get(cur, 0) + 1
    return res

def count_div_brute(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if sum(nums[i:j + 1]) % k == 0)

assert count_div_subarrays_sol([4, 5, 0, -2, -3, 1], 5) == 7
random.seed(3)
for _ in range(500):
    a = [random.randint(-10, 10) for _ in range(random.randint(1, 10))]
    k = random.randint(1, 7)
    assert count_div_subarrays_sol(a, k) == count_div_brute(a, k)
print("Bài 3: OK")

Bài 3: OK


In [29]:
# Bài 4
def min_subarray_with_range_sol(nums, p):
    n = len(nums)
    k = sum(nums) % p
    if k == 0:
        return (0, -1, -1)
    last = {0: -1}; cur = 0
    best = (n, -1, -1)
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        need = (cur - k) % p
        if need in last and j - last[need] < best[0]:
            best = (j - last[need], last[need] + 1, j)
        last[cur] = j
    return (-1, -1, -1) if best[0] == n else best

assert min_subarray_with_range_sol([3, 1, 4, 2], 6) == (1, 2, 2)
assert min_subarray_with_range_sol([6, 3, 5, 2], 9) == (2, 2, 3)
assert min_subarray_with_range_sol([1, 2, 3], 3) == (0, -1, -1)
assert min_subarray_with_range_sol([1, 2, 3], 7) == (-1, -1, -1)
random.seed(4)
for _ in range(1000):
    a = [random.randint(1, 20) for _ in range(random.randint(1, 10))]
    p = random.randint(1, 12)
    L, l, r = min_subarray_with_range_sol(a, p)
    assert L == min_subarray_brute(a, p)
    if L > 0:
        assert r - l + 1 == L and (sum(a) - sum(a[l:r + 1])) % p == 0
print("Bài 4: OK")

Bài 4: OK


In [30]:
# Bài 5
def shortest_divisible_subarray_sol(nums, p):
    n = len(nums)
    last = {0: -1}; cur = 0; ans = n + 1
    for j, x in enumerate(nums):
        cur = (cur + x) % p
        if cur in last:
            ans = min(ans, j - last[cur])
        last[cur] = j
    return -1 if ans == n + 1 else ans

def shortest_div_brute(nums, p):
    n = len(nums); best = -1
    for i in range(n):
        for j in range(i, n):
            if sum(nums[i:j + 1]) % p == 0 and (best == -1 or j - i + 1 < best):
                best = j - i + 1
    return best

assert shortest_divisible_subarray_sol([1, 2, 3], 6) == 3
assert shortest_divisible_subarray_sol([1, 1], 5) == -1
random.seed(5)
for _ in range(1000):
    a = [random.randint(1, 20) for _ in range(random.randint(1, 10))]
    p = random.randint(1, 15)
    assert shortest_divisible_subarray_sol(a, p) == shortest_div_brute(a, p)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đoạn ngắn nhất có tổng ≡ k (mod p)":**

```python
k = sum(nums) % p
if k == 0: return 0
last, cur, ans = {0: -1}, 0, n
for j, x in enumerate(nums):
    cur = (cur + x) % p
    need = (cur - k) % p
    if need in last: ans = min(ans, j - last[need])
    last[cur] = j            # gần nhất cho "ngắn nhất"
return -1 if ans == n else ans
```

**Checklist:**
- [ ] Đã xử lý `k == 0` → trả `0`?
- [ ] Map khởi tạo `{0: -1}` (tiền tố rỗng)?
- [ ] `need` có `% p` không?
- [ ] Ghi đè `last[cur] = j` (gần nhất) — không dùng `setdefault`?
- [ ] Tra trước, ghi sau?
- [ ] Trả `-1` khi `ans == n` (không được xóa cả mảng)?
- [ ] Đã thử: 1 phần tử, `p = 1`, `p` > tổng, đáp án là tiền tố / hậu tố, chỉ cả mảng thỏa?